# Homework — Protein structures and the PDB

You've spent the last few weeks learning **pandas**: DataFrames, filtering rows,
`.mean()`, `.value_counts()`. This homework hands you a real protein structure and asks
you to notice one big idea: **a protein structure is just a pandas DataFrame of atom
coordinates.** Everything you already know applies.

**How this works**
1. Run the setup cell (it installs `biopandas` — a few seconds).
2. Read each short section and run its example — outputs are shown as `# ...` comments.
3. Answer the numbered **Problems** in the empty cells. Watch for the **Reasoning** tag:
   those want an *explanation* (write it as a `# comment`), not just a number — they count
   just as much as the code.
4. At the end there's a **check-in value** — type it into the CHEM 438 app to finish.

Run every cell top to bottom.

In [ ]:
# Setup — run this first
!pip install biopandas -q
from biopandas.pdb import PandasPdb
import numpy as np
import pandas as pd
print("Ready.")

## 1. What is a PDB file, and how do we load one?

Scientists determine the 3D shape of a protein (X-ray crystallography, cryo-EM, NMR) and
deposit it in the **Protein Data Bank (PDB)**. Each structure gets a 4-character ID, like
`1AKI` — hen egg-white **lysozyme**, a small, well-behaved enzyme.

A **PDB file** is mostly a long list of `ATOM` records. Each line is one atom and carries
its **x, y, z coordinates in Angstroms (A)**, its name (`CA`, `N`, `O`, ...), and which
residue (amino acid) it belongs to. A structure is just a table of atoms with positions.

`PandasPdb().fetch_pdb('1AKI')` downloads it; the atom records live in `ppdb.df['ATOM']`.

In [ ]:
ppdb = PandasPdb().fetch_pdb('1AKI')
atoms = ppdb.df['ATOM']
print(type(atoms))       # <class 'pandas.core.frame.DataFrame'>
print(atoms.shape)       # (1001, 21)  -> 1001 atoms, 21 columns

## 2. The atom table is just a DataFrame

There's the punchline: `atoms` is a **`pandas.DataFrame`** — the same object you filtered
and averaged all semester. The columns you'll use most:

- `atom_name` — e.g. `N`, `CA`, `C`, `O`, `CB`
- `residue_name` — the amino acid, e.g. `LYS`, `ARG`, `GLY`
- `residue_number` — which residue in the chain (1, 2, 3, ...)
- `chain_id` — which chain (`A`, `B`, ...)
- `x_coord`, `y_coord`, `z_coord` — position in Angstroms
- `element_symbol` — `C`, `N`, `O`, `S`

In [ ]:
# One atom = one row. Here are the first three atoms:
print(atoms[['atom_name', 'residue_name', 'residue_number', 'chain_id',
             'x_coord', 'y_coord', 'z_coord', 'element_symbol']].head(3))
# atom_name residue_name  residue_number chain_id  x_coord  y_coord  z_coord element_symbol
#         N          LYS               1        A    35.365   22.342  -11.980              N
#        CA          LYS               1        A    35.892   21.073  -11.427              C
#         C          LYS               1        A    34.741   20.264  -10.844              C

## 3. Counting and filtering — exactly like the pandas homework

Rows are atoms, so `len(atoms)` counts atoms. Counting *residues* is different: many
atoms share one `residue_number`, so you count **unique** values with `.nunique()`.

Filtering is the same boolean-mask move you already know: `atoms[atoms['col'] == value]`.

In [ ]:
print("atoms:   ", len(atoms))                        # 1001
print("residues:", atoms['residue_number'].nunique())  # 129

carbons = atoms[atoms['element_symbol'] == 'C']        # a boolean-mask filter
print("carbon atoms:", len(carbons))                   # 613

ca = atoms[atoms['atom_name'] == 'CA']                 # the backbone alpha-carbons
print("CA atoms:", len(ca))                            # 129  (one per residue!)

## 4. The geometric center

Where is the "middle" of the protein? Average every atom's x, then every y, then every z.
The point `(x_mean, y_mean, z_mean)` is the **geometric center** (the centroid) — just
`.mean()` on three columns.

In [ ]:
cx = atoms['x_coord'].mean()
cy = atoms['y_coord'].mean()
cz = atoms['z_coord'].mean()
print("center:", round(cx, 1), round(cy, 1), round(cz, 1))   # 27.6 25.0 0.2

## 5. Distance between two atoms

Each atom is a point `(x, y, z)`, so the distance between two atoms is the ordinary 3D
distance formula:

$$d = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2 + (z_1-z_2)^2}$$

Below: the first two atoms of the protein — the backbone **N** and **CA** of residue 1.

In [ ]:
a = atoms.iloc[0]   # N of residue 1
b = atoms.iloc[1]   # CA of residue 1
d = np.sqrt((a['x_coord']-b['x_coord'])**2 +
            (a['y_coord']-b['y_coord'])**2 +
            (a['z_coord']-b['z_coord'])**2)
print(a['atom_name'], "to", b['atom_name'], "=", round(d, 2), "Angstrom")  # N to CA = 1.48 Angstrom

## Problems

Write your answer in the empty cell under each problem. For **Reasoning** problems, put
your explanation in a `# comment` right in the code cell.

**Problem 1 — Count.** Using the same `atoms` DataFrame, print how many **nitrogen**
atoms are in the structure. It's the exact filter-then-count move from Section 3, with a
different element.

**Problem 2 — Reasoning (connect-back).** In the pandas homework you counted the rows
that matched a condition. The atom records are *that same kind of DataFrame*. In one or
two sentences, describe **in words** how you would count how many atoms are **oxygen**:
which column you'd look at, what value you'd compare it to, and what pandas hands back.
Then write the code and confirm you were right.

**Problem 3 — Interpret.** Section 3 found **1001 atoms** but only **129 residues** —
roughly 8 atoms per residue. In a comment, explain **why there are so many more atoms
than residues.** (Hint: think about what a single amino acid is actually built from.)

**Problem 4 — Predict, then check.** `1AKI` contains only chain `A`. *Before running
anything*, predict: if you filtered the DataFrame to just chain `A` and recomputed the
geometric center, would it **move** compared to the center in Section 4? Write your
prediction as a comment. Then filter with `atoms[atoms['chain_id'] == 'A']`, recompute the
center, and confirm. In one line, explain **why** it came out that way.

**Problem 5 — Compute-and-reason.** The N-CA distance in Section 5 was **1.48 A**, a
typical covalent bond length. Now compute the distance between the **N of residue 1**
(`atoms.iloc[0]`) and the **CA of residue 2**. Get residue 2's CA row with
`atoms[(atoms['residue_number'] == 2) & (atoms['atom_name'] == 'CA')].iloc[0]`, then print
the distance. In a comment, answer: is that distance consistent with a *direct* chemical
bond between those two atoms, or are they just near each other? Roughly what distance would
tell you two atoms are **not** bonded?

**Problem 6 — Why.** We got the center by **averaging** the coordinates of every atom.
In a comment, explain what that average silently **assumes about the atoms** — does it
treat a heavy sulfur atom the same as a light carbon? What would you need to change to
compute a true *center of mass* instead of a geometric center?

**Problem 7 — Filter to one residue type.** Print how many atoms belong to **glycine**
residues (`residue_name == 'GLY'`). Then, in a comment, say whether that atom count is the
same as the *number of glycine residues* — and if not, what you'd compute instead to get
the number of glycines. (Section 3 showed the trick.)

**Problem 8 — Concept check (multiple choice).** When you load `1AKI` with biopandas,
what is `ppdb.df['ATOM']`? Set `answer` to the letter you believe is correct.

- **A.** A picture (image file) of the protein
- **B.** A pandas DataFrame with one row per atom and its 3D coordinates
- **C.** A single text string of the whole PDB file
- **D.** A list of amino-acid names only, with no coordinates

In [ ]:
answer = "?"   # replace ? with A, B, C, or D
print(answer)

## Check in

Run the cell below. It loads `1AKI` the same way every time and prints **one number**: the
count of **carbon atoms** in the structure — the filter-then-count you practiced in
Section 3 and Problem 1.

**Type that number into the CHEM 438 app to mark this homework complete.**

In [ ]:
ppdb = PandasPdb().fetch_pdb('1AKI')
atoms = ppdb.df['ATOM']
carbon_count = len(atoms[atoms['element_symbol'] == 'C'])
print("Your check-in value:", carbon_count)